# SeedSec Rwanda: Corn Seed Image Classification Training Notebook
This Jupyter notebook contains the training script to train a **ResNet50** classification model for classifying **Corn Seed Defects** (Pure, Broken, Discolored, Silk Cut) using the Kaggle dataset. It handles mounting Google Drive, setting up API credentials, downloading the dataset, splitting it into training, validation, and test subsets, training the model with PyTorch/torchvision, validating performance (F1-score, precision, recall, and plotting curves), and exporting the weights to quantized TFLite format.

## Step 1: Set Kaggle Token & Mount Google Drive

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set Kaggle token
os.environ['KAGGLE_API_TOKEN'] = 'YOUR_KAGGLE_TOKEN_HERE'

# Install Kaggle
!pip install -q kaggle

# Create target folder in Drive
!mkdir -p "/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification"

# Download dataset
!kaggle datasets download -d linxvandu/corn-seed-image-classification \
  -p "/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification"

# Unzip the file
!unzip -o "/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification/corn-seed-image-classification.zip" \
  -d "/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification"

# Check files
!ls -lah "/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification"

## Step 2: Explore Dataset Directory Structure
Before splitting, we must discover the actual folder layout that the Kaggle zip extracted. This cell walks the directory tree and prints every folder and file count so we know where the class images live.

In [ ]:
import os
from pathlib import Path

base_dir = Path("/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification")

print(f"Base directory exists: {base_dir.exists()}")
print(f"Contents of base directory:")
for item in sorted(base_dir.iterdir()):
    if item.is_dir():
        n_files = len(list(item.rglob('*')))
        print(f"  [DIR]  {item.name}/ ({n_files} items inside)")
    else:
        print(f"  [FILE] {item.name} ({item.stat().st_size / 1024:.1f} KB)")

print("\n--- Full recursive directory tree (first 50 entries) ---")
for i, p in enumerate(sorted(base_dir.rglob('*'))):
    if i >= 50:
        print(f"  ... and more (stopped at 50)")
        break
    rel = p.relative_to(base_dir)
    tag = '[DIR]' if p.is_dir() else '[IMG]' if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'} else '[FILE]'
    print(f"  {tag} {rel}")

## Step 3: Organize Images & Train/Val/Test Split
Scans the directory structure of the extracted Kaggle dataset, identifies all class folders representing seed qualities, and splits the images into train (70%), validation (15%), and test (15%) groups.

**Important:** This cell automatically detects the dataset layout whether the images are:
- Flat: `dataset/ClassName/image.jpg`
- Pre-split: `dataset/train/ClassName/image.jpg` and `dataset/test/ClassName/image.jpg`
- Nested: `dataset/some_folder/ClassName/image.jpg`

In [ ]:
import os
import random
import shutil
from pathlib import Path
from collections import defaultdict

# ============================================================
# CONFIGURATION — Point this to your actual extraction folder
# ============================================================
base_dir = Path("/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification")
split_target_dir = Path("/content/corn_splits")

# Clean previous splits if re-running
if split_target_dir.exists():
    shutil.rmtree(split_target_dir)

splits = ["train", "val", "test"]
for s in splits:
    (split_target_dir / s).mkdir(parents=True, exist_ok=True)

# ============================================================
# STEP A: Find ALL image files recursively
# ============================================================
valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
all_images = [
    p for p in base_dir.rglob('*')
    if p.is_file() and p.suffix.lower() in valid_exts
]
print(f"Total image files found: {len(all_images)}")

if len(all_images) == 0:
    raise RuntimeError(
        f"No image files found under {base_dir}.\n"
        f"Please verify Step 1 completed successfully and check the path."
    )

# ============================================================
# STEP B: Determine the class name for each image
# The class is the LEAF directory that contains the image.
# If that leaf name is a split name (train/test/val), go one
# level deeper — the class is the image's grandparent.
# ============================================================
SPLIT_NAMES = {'train', 'test', 'val', 'valid', 'validation'}

class_to_images = defaultdict(list)
skipped = 0

for img_path in all_images:
    parent = img_path.parent
    class_name = parent.name

    # Skip images sitting directly in the base_dir root (no class folder)
    if parent == base_dir:
        skipped += 1
        continue

    # If parent is a split folder (train/test), skip — class is unknown
    if class_name.lower() in SPLIT_NAMES:
        skipped += 1
        continue

    # Accept this as a valid class
    class_to_images[class_name].append(img_path)

print(f"Skipped {skipped} images (no class folder or in root)")
print(f"Found {len(class_to_images)} class folders:")
for cls, imgs in sorted(class_to_images.items()):
    print(f"  - {cls}: {len(imgs)} images")

if len(class_to_images) == 0:
    # Last resort: maybe images are in base_dir/subfolder/ without class dirs
    print("\n⚠️ No class folders detected. Listing parent folders of all images:")
    parents = set(img.parent for img in all_images)
    for p in sorted(parents):
        print(f"  {p} ({len(list(p.iterdir()))} files)")
    raise RuntimeError("Cannot determine class folders. Check directory structure above.")

# ============================================================
# STEP C: Perform the 70/15/15 split
# ============================================================
random.seed(42)

for class_name, imgs in class_to_images.items():
    random.shuffle(imgs)

    for s in splits:
        (split_target_dir / s / class_name).mkdir(parents=True, exist_ok=True)

    n = len(imgs)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    for img in imgs[:n_train]:
        shutil.copy2(str(img), split_target_dir / "train" / class_name / img.name)
    for img in imgs[n_train:n_train + n_val]:
        shutil.copy2(str(img), split_target_dir / "val" / class_name / img.name)
    for img in imgs[n_train + n_val:]:
        shutil.copy2(str(img), split_target_dir / "test" / class_name / img.name)

print("\n✅ Splits successfully configured:")
for s in splits:
    count = len(list((split_target_dir / s).rglob('*.*')))
    print(f"  - {s}: {count} images")

## Step 4: Define PyTorch Dataset & Transforms
Configures image augmentations (rotations, flips, color normalization) matching **ResNet50** inputs ($224 \times 224$ pixels) and loads them into DataLoader generators.

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Standard normalizations for ResNet pre-trained weights
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder("/content/corn_splits/train", transform=train_transform)
val_dataset = datasets.ImageFolder("/content/corn_splits/val", transform=val_test_transform)
test_dataset = datasets.ImageFolder("/content/corn_splits/test", transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset.classes
print(f"Target class names mapping ({len(class_names)}): {class_names}")

## Step 5: Initialize ResNet50 & Define Optimizer/Loss
Loads pre-trained **ResNet50** weights, freezes the feature layers, and overrides the fully connected (`fc`) linear layer to output our target class predictions.

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware accelerator: {device}")

# Load ResNet50 with pre-trained weights
model = resnet50(weights=ResNet50_Weights.DEFAULT)

# Freeze feature extraction layers for transfer learning
for param in model.parameters():
    param.requires_grad = False

# Swap classification output fc layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

print(model.fc)

## Step 6: Train the Model
Runs model training for 15 epochs, saving the best weight checkpoint based on validation accuracy.

In [ ]:
import time

epochs = 15
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

best_val_acc = 0.0
weights_dir = "/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification/weights"
os.makedirs(weights_dir, exist_ok=True)
best_model_path = os.path.join(weights_dir, "best_resnet50_corn.pth")

print("Starting training loop...")
for epoch in range(epochs):
    t0 = time.time()
    model.train()
    train_loss, train_corrects = 0.0, 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, preds = torch.max(outputs, 1)
        train_loss += loss.item() * inputs.size(0)
        train_corrects += torch.sum(preds == labels.data)
        
    epoch_loss = train_loss / len(train_dataset)
    epoch_acc = train_corrects.double().item() / len(train_dataset)
    
    # Validation phase
    model.eval()
    val_loss, val_corrects = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)
            
    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_acc = val_corrects.double().item() / len(val_dataset)
    
    history["train_loss"].append(epoch_loss)
    history["train_acc"].append(epoch_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    
    t_elapsed = time.time() - t0
    print(f"Epoch {epoch+1}/{epochs} ({t_elapsed:.1f}s) -> "
          f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")
    
    # Save best weight checkpoints
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"   ---> Saved new best checkpoint with validation accuracy: {best_val_acc:.4f}")

## Step 7: Plot Training History
Generates plots showing loss and accuracy curves for validation.

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, epochs + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Training Loss", color='blue')
plt.plot(epochs_range, history["val_loss"], label="Validation Loss", color='orange')
plt.title("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Training Accuracy", color='blue')
plt.plot(epochs_range, history["val_acc"], label="Validation Accuracy", color='orange')
plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## Step 8: Evaluate Classification Metrics & Generate Confusion Matrix
Computes detailed classification metrics (F1-score, Precision, Recall) per class and plots a Confusion Matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Load best model weights
model.load_state_dict(torch.load(best_model_path))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Print Classification Report
print("\n=================== TEST SUBSET EVALUATION REPORT ===================")
print(classification_report(all_labels, all_preds, target_names=class_names))
print("======================================================================\n")

# Plot Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Corn Seed Defect Classification")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Step 9: Export Model to TFLite
Converts the ResNet50 PyTorch weights to ONNX format, then converts it to TensorFlow, and compiles it down to a FP16-quantized TensorFlow Lite model to support offline local device diagnostics.

In [ ]:
# Install ONNX and TensorFlow converter utilities including NVIDIA graphsurgeon index
!pip install onnx onnx2tf tensorflow onnx-graphsurgeon --index-url https://pypi.ngc.nvidia.com -q

dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path = "/content/corn_seed_classifier.onnx"

# Export PyTorch model to standard ONNX representation
torch.onnx.export(
    model.cpu(),
    dummy_input.cpu(),
    onnx_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input_img'],
    output_names=['output_scores'],
    dynamic_axes={'input_img': {0: 'batch_size'}, 'output_scores': {0: 'batch_size'}}
)
print(f"ONNX model exported to: {onnx_path}")

# Convert ONNX to TensorFlow format using onnx2tf with FP16 quantization
tf_dir = "/content/tf_model"
!onnx2tf -i "$onnx_path" -o "$tf_dir" -pqt -qt float16 --non_verbose

# Find generated file and copy it to Google Drive
tflite_out_path = "/content/drive/MyDrive/KaggleDatasets/corn-seed-image-classification/corn_seed_resnet50_fp16.tflite"
generated_tflite = "/content/tf_model/corn_seed_classifier_float16.tflite"

import shutil
if os.path.exists(generated_tflite):
    shutil.copy(generated_tflite, tflite_out_path)
    print(f"Successfully exported quantized TFLite classifier model to: {tflite_out_path}")
else:
    # Fallback search
    import glob
    tflite_files = glob.glob("/content/tf_model/**/*.tflite", recursive=True)
    if tflite_files:
        shutil.copy(tflite_files[0], tflite_out_path)
        print(f"Successfully exported quantized TFLite model from fallback search: {tflite_out_path}")
    else:
        print("⚠️ Warning: TFLite file could not be located in conversion directory.")